# Accommodation and Vergence — Schor Dual-Interaction Model

Demonstrates the coupled accommodation–vergence system implemented in
`accommodation.py` and `vergence.py`, following Schor (1979) and Schor & Ciuffreda (1983).

---

## Background

The primate oculomotor system maintains both **focus** (accommodation) and **alignment**
(vergence) simultaneously when viewing targets at different distances.
These two systems are tightly coupled via two cross-links:

### AC/A — Accommodative Convergence / Accommodation

When the lens accommodates to focus a near target, a conjugate vergence signal is
reflexively generated:

$$\text{vergence drive} = \text{AC/A} \times 0.573\ (\text{deg/pd}) \times x_{\text{acc}}\ (\text{D})$$

- **AC/A ratio** ≈ 4–6 pd/D in healthy adults [Morgan 1944; Alpern 1958]
- This drive acts **independent of binocular disparity** — critical for re-fusion in
  intermittent exotropia (IXT), where disparity information is absent when one eye drifts
  outside Panum's fusional area.
- At 6 m (0.17 D accommodation): AC/A drive ≈ 5 × 0.573 × 0.17 ≈ **0.5°** — very small.
- At 15 cm (6.67 D): AC/A drive ≈ 5 × 0.573 × 6.67 ≈ **19°** — major contribution.

### CA/C — Convergence Accommodation / Convergence

When vergence is driven by disparity, it co-drives the lens:

$$A_{\text{CA/C}} = \text{CA/C} \times x_{\text{verg}} / 0.573$$

- **CA/C ratio** ≈ 0.3–0.5 D/pd [Fincham & Walton 1957]
- Reduces blur during vergence eye movements; stabilises the coupled loop.

### Dual-component accommodation (Schor 1979)

$$\dot{x}_{\text{fast}} = -x_{\text{fast}}/\tau_{\text{fast}} + K_{\text{fast}} \cdot e_{\text{blur}}$$
$$\dot{x}_{\text{slow}} = -x_{\text{slow}}/\tau_{\text{slow}} + K_{\text{slow}} \cdot e_{\text{blur}}$$
$$e_{\text{blur}} = d_{\text{acc}} - (x_{\text{fast}} + x_{\text{slow}}) - \text{CA/C} \cdot x_{\text{verg}}/0.573$$

- **Fast phasic** component ($\tau$ ≈ 0.3 s): quick response; fatigues with sustained blur.
- **Slow tonic** component ($\tau$ ≈ 30 s): integrates residual blur; models lens adaptation.

### Key references

| Reference | Contribution |
|-----------|-------------|
| Schor (1979) *Vision Res* 19:1359 | Dual slow/fast interaction model |
| Schor & Ciuffreda (1983) *Vergence Eye Movements* | Definitive textbook treatment |
| Rashbass & Westheimer (1961) *J Physiol* 159:361 | Vergence TC ~160 ms |
| Fincham & Walton (1957) *J Physiol* 137:488 | CA/C cross-link |
| Morgan (1944) *Am J Optom* 21:301 | AC/A normative data |
| Hung & Semmlow (1980) *IEEE Trans Biomed Eng* 27:722 | Vergence dynamics |
| Semmlow et al. (1986) *Invest Ophthalmol* 27:558 | Tonic vergence TC ~5–7 s |

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from oculomotor.sim.simulator import (
    PARAMS_DEFAULT, with_brain, simulate,
    _IDX_VERG, _IDX_ACC,
)
from oculomotor import __version__

print(f'oculomotor {__version__}')

THETA = PARAMS_DEFAULT
DT    = 0.001
IPD   = THETA.sensory.ipd   # m

print(f'\nAccommodation parameters:')
print(f'  tau_acc_fast = {THETA.brain.tau_acc_fast} s')
print(f'  tau_acc_slow = {THETA.brain.tau_acc_slow} s')
print(f'  K_acc_fast   = {THETA.brain.K_acc_fast} /s')
print(f'  K_acc_slow   = {THETA.brain.K_acc_slow} /s')
print(f'  AC/A         = {THETA.brain.AC_A} pd/D')
print(f'  CA/C         = {THETA.brain.CA_C} D/pd')
print(f'\nVergence parameters:')
print(f'  K_verg       = {THETA.brain.K_verg}')
print(f'  tau_verg     = {THETA.brain.tau_verg} s')

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size']  = 9


def verg_demand_deg(z_m):
    """Geometric vergence demand at midline depth z_m (m)."""
    return np.degrees(2.0 * np.arctan(IPD / 2.0 / np.asarray(z_m)))


def acc_demand_D(z_m):
    """Accommodation demand for target at depth z_m (m)."""
    return 1.0 / np.asarray(z_m)


def ax_fmt(ax, ylabel='', xlabel='', ylim=None):
    ax.axhline(0, color='gray', lw=0.4)
    ax.grid(True, alpha=0.2)
    if ylabel: ax.set_ylabel(ylabel, fontsize=8)
    if xlabel: ax.set_xlabel(xlabel, fontsize=8)
    if ylim:   ax.set_ylim(ylim)
    ax.tick_params(labelsize=7)


C = {
    'L':     '#2166ac',
    'R':     '#d6604d',
    'verg':  '#1b7837',
    'fast':  '#e08214',
    'slow':  '#762a83',
    'total': '#000000',
    'dem':   '#999999',
    'aca':   '#d73027',
    'cac':   '#4575b4',
}

---
## Demo 1 — Accommodation dynamics: fast + slow components

Target steps from far (3 m, 0.33 D) to near (0.25 m, 4 D) at t = 2 s, then back to far at t = 10 s.

**Expected behaviour:**
- **Fast component** rises quickly (TC ≈ 0.3 s), then partially decays as the slow component builds.
- **Slow component** integrates residual blur over ~30 s; doesn't fully load in 8 s so fast remains elevated.
- **Total accommodation** tracks the demand with a latency determined primarily by the fast component.
- On return to far: fast component relaxes quickly; slow component unloads slowly.

In [ ]:
Z_FAR  = 3.0    # m
Z_NEAR = 0.25   # m
T_END1 = 18.0

t1   = jnp.arange(0.0, T_END1, DT)
T1   = len(t1)
t1np = np.array(t1)

z1   = jnp.where(t1 < 2.0, Z_FAR, jnp.where(t1 < 10.0, Z_NEAR, Z_FAR))
pt1  = jnp.stack([jnp.zeros(T1), jnp.zeros(T1), z1], axis=1)

print('Simulating Demo 1...')
st1 = simulate(
    THETA, t1,
    p_target_array      = pt1,
    scene_present_array = jnp.ones(T1),
    return_states       = True,
    key                 = jax.random.PRNGKey(0),
)
print('Done.')

z1np      = np.array(z1)
acc_dem   = acc_demand_D(z1np)
x_acc1    = np.array(st1.brain[:, _IDX_ACC])      # (T, 2): [fast, slow]
x_fast1   = x_acc1[:, 0]
x_slow1   = x_acc1[:, 1]
x_total1  = x_fast1 + x_slow1

fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
fig.suptitle('Demo 1 — Accommodation dynamics: fast + slow components', fontweight='bold')

# Depth
ax = axes[0]
ax.plot(t1np, z1np, color='k', lw=1.5)
ax.axvline(2.0,  color='gray', lw=0.7, ls='--')
ax.axvline(10.0, color='gray', lw=0.7, ls='--')
ax_fmt(ax, ylabel='Target depth (m)')
ax.set_ylim(0, 3.5)
ax.set_title(f'Target: {Z_FAR} m → {Z_NEAR} m → {Z_FAR} m', fontsize=8)

# Accommodation demand vs response (with components)
ax = axes[1]
ax.plot(t1np, acc_dem,  color=C['dem'],   lw=1.2, ls='--', label='demand (1/z D)')
ax.plot(t1np, x_fast1,  color=C['fast'],  lw=1.2, ls='-',  label='x_fast (phasic, τ≈0.3s)')
ax.plot(t1np, x_slow1,  color=C['slow'],  lw=1.2, ls='-',  label='x_slow (tonic, τ≈30s)')
ax.plot(t1np, x_total1, color=C['total'], lw=1.8, ls='-',  label='x_acc total')
ax.axvline(2.0,  color='gray', lw=0.7, ls='--')
ax.axvline(10.0, color='gray', lw=0.7, ls='--')
ax.legend(fontsize=7, loc='upper right')
ax_fmt(ax, ylabel='Accommodation (D)')
ax.set_title('Accommodation components — fast responds first, slow integrates residual blur', fontsize=8)

# Residual blur error
blur_err1 = acc_dem - x_total1
ax = axes[2]
ax.plot(t1np, blur_err1, color='#8c510a', lw=1.4, label='blur error (demand − total)')
ax.axvline(2.0,  color='gray', lw=0.7, ls='--')
ax.axvline(10.0, color='gray', lw=0.7, ls='--')
ax.legend(fontsize=7)
ax_fmt(ax, ylabel='Blur error (D)', xlabel='Time (s)')
ax.set_title('Residual blur error — driven to zero by dual-component integration', fontsize=8)

fig.tight_layout()
plt.show()

print(f'\nNear (steady state ~9s):  demand={acc_demand_D(Z_NEAR):.2f}D  '
      f'total={x_total1[int(9.0/DT)]:.2f}D  '
      f'(fast={x_fast1[int(9.0/DT)]:.2f}  slow={x_slow1[int(9.0/DT)]:.2f})')

---
## Demo 2 — AC/A drive: accommodation-driven vergence

Compare vergence responses for **two targets at different distances**:
- **Near** (0.25 m): large accommodation (4 D) → large AC/A drive
- **Far** (3 m): small accommodation (0.33 D) → small AC/A drive

The AC/A component is the *accommodative* contribution to vergence — vergence from
accommodation, not from disparity. Both components act in the healthy system.

**AC/A drive magnitude** = AC_A × 0.573 × x_acc_total (deg)

In [ ]:
depths_to_test = [3.0, 1.0, 0.5, 0.25, 0.15]
T_STEP = 8.0

results = []
print('Simulating AC/A demo (one run per depth)...')

for z_m in depths_to_test:
    t_s  = jnp.arange(0.0, T_STEP, DT)
    T_s  = len(t_s)
    pt_s = jnp.tile(jnp.array([0.0, 0.0, float(z_m)]), (T_s, 1))
    st_s = simulate(
        THETA, t_s,
        p_target_array      = pt_s,
        scene_present_array = jnp.ones(T_s),
        return_states       = True,
        key                 = jax.random.PRNGKey(0),
    )
    x_acc_s  = np.array(st_s.brain[:, _IDX_ACC])
    x_verg_s = np.array(st_s.brain[:, _IDX_VERG])
    ep_L_s   = np.array(st_s.plant[:, 0])
    ep_R_s   = np.array(st_s.plant[:, 3])
    verg_s   = ep_L_s - ep_R_s
    aca_s    = THETA.brain.AC_A * 0.5729 * (x_acc_s[:, 0] + x_acc_s[:, 1])
    results.append(dict(
        z=z_m, t=np.array(t_s),
        acc_total=x_acc_s[:, 0]+x_acc_s[:, 1],
        verg=verg_s, aca=aca_s,
        dem_verg=verg_demand_deg(z_m),
        dem_acc=acc_demand_D(z_m),
    ))
    print(f'  z={z_m:.2f}m  acc_dem={acc_demand_D(z_m):.2f}D  '
          f'AC/A drive@SS={aca_s[-1]:.1f}deg  verg@SS={verg_s[-1]:.1f}deg  '
          f'(demand={verg_demand_deg(z_m):.1f}deg)')

print('Done.')

# Plot steady-state AC/A contribution vs depth
z_arr  = np.array([r['z'] for r in results])
aca_ss = np.array([r['aca'][-1] for r in results])
vss    = np.array([r['verg'][-1] for r in results])
dv_ss  = np.array([r['dem_verg'] for r in results])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Demo 2 — AC/A drive as a function of viewing distance', fontweight='bold')

ax = axes[0]
ax.plot(z_arr, dv_ss, 'o--', color=C['dem'],  lw=1.2, ms=6, label='Geometric vergence demand')
ax.plot(z_arr, vss,   's-',  color=C['verg'], lw=1.5, ms=7, label='Actual vergence (model SS)')
ax.plot(z_arr, aca_ss,'D:',  color=C['aca'],  lw=1.2, ms=6, label='AC/A component alone')
ax.set_xlabel('Target depth (m)', fontsize=8)
ax.set_ylabel('Vergence angle (deg)', fontsize=8)
ax.set_title('Steady-state vergence vs. depth\nAC/A contribution grows with accommodation demand', fontsize=8)
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)
ax.invert_xaxis()

ax = axes[1]
colors = plt.cm.plasma(np.linspace(0.1, 0.85, len(results)))
for r, col in zip(results, colors):
    ax.plot(r['t'], r['aca'], color=col, lw=1.5, label=f'z={r["z"]}m ({r["dem_acc"]:.1f}D)')
ax.set_xlabel('Time (s)', fontsize=8)
ax.set_ylabel('AC/A drive (deg)', fontsize=8)
ax.set_title('AC/A drive dynamics — larger at near, rises with accommodation TC', fontsize=8)
ax.legend(fontsize=7, loc='upper right')
ax.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

---
## Demo 3 — CA/C coupling: vergence-driven accommodation

The CA/C (Convergence Accommodation / Convergence) cross-link co-drives the lens
when vergence is commanded by disparity:

$$A_{\text{CA/C}} = \text{CA/C} \times x_{\text{verg}} / 0.573 \approx 0.4 \times x_{\text{verg}} / 0.573 \text{ D}$$

This means a **10° convergence** drives ~**7 pd** of vergence → CA/C drives ~0.4×7 = **2.8 D**
of accommodation — keeping the lens roughly in focus as vergence changes.

**Demo**: Target on the midline steps in depth (far→near→far). We show how much of the
accommodation comes from direct blur feedback vs. CA/C from the vergence state.

In [ ]:
Z_FAR3  = 2.0
Z_NEAR3 = 0.3
T_END3  = 15.0

t3   = jnp.arange(0.0, T_END3, DT)
T3   = len(t3)
t3np = np.array(t3)

z3   = jnp.where(t3 < 2.0, Z_FAR3, jnp.where(t3 < 9.0, Z_NEAR3, Z_FAR3))
pt3  = jnp.stack([jnp.zeros(T3), jnp.zeros(T3), z3], axis=1)

print('Simulating Demo 3 (CA/C coupling)...')
st3 = simulate(
    THETA, t3,
    p_target_array      = pt3,
    scene_present_array = jnp.ones(T3),
    return_states       = True,
    key                 = jax.random.PRNGKey(0),
)
print('Done.')

z3np     = np.array(z3)
x_acc3   = np.array(st3.brain[:, _IDX_ACC])
x_verg3  = np.array(st3.brain[:, _IDX_VERG])
ep_L3    = np.array(st3.plant[:, 0])
ep_R3    = np.array(st3.plant[:, 3])
vergence3 = ep_L3 - ep_R3

acc_total3  = x_acc3[:, 0] + x_acc3[:, 1]
cac_drive3  = THETA.brain.CA_C * (x_verg3[:, 0] / 0.5729)   # D: CA/C contribution
aca_drive3  = THETA.brain.AC_A * 0.5729 * acc_total3          # deg: AC/A vergence drive
acc_dem3    = acc_demand_D(z3np)
verg_dem3   = verg_demand_deg(z3np)

fig, axes = plt.subplots(4, 1, figsize=(13, 12), sharex=True)
fig.suptitle('Demo 3 — CA/C coupling: vergence co-drives accommodation', fontweight='bold')

ax = axes[0]
ax.plot(t3np, z3np, color='k', lw=1.5)
ax.axvline(2.0, color='gray', lw=0.7, ls='--')
ax.axvline(9.0, color='gray', lw=0.7, ls='--')
ax_fmt(ax, ylabel='Target depth (m)')
ax.set_ylim(0, 2.5)
ax.set_title(f'Target: {Z_FAR3} m → {Z_NEAR3} m → {Z_FAR3} m', fontsize=8)

ax = axes[1]
ax.plot(t3np, verg_dem3,  color=C['dem'],  lw=1.2, ls='--', label='demand (geometry)')
ax.plot(t3np, vergence3,  color=C['verg'], lw=1.5,          label='vergence (L−R plant)')
ax.plot(t3np, x_verg3[:, 0], color=C['slow'], lw=1.0, ls=':', label='x_verg state')
ax.axvline(2.0, color='gray', lw=0.7, ls='--')
ax.axvline(9.0, color='gray', lw=0.7, ls='--')
ax.legend(fontsize=7)
ax_fmt(ax, ylabel='Vergence (deg)')
ax.set_title('Vergence — drives CA/C accommodation', fontsize=8)

ax = axes[2]
ax.plot(t3np, acc_dem3,   color=C['dem'],   lw=1.2, ls='--', label='blur demand (1/z)')
ax.plot(t3np, acc_total3, color=C['total'], lw=1.8,          label='x_acc total')
ax.plot(t3np, cac_drive3, color=C['cac'],   lw=1.2, ls='-.',  label='CA/C drive (vergence→lens)')
ax.axvline(2.0, color='gray', lw=0.7, ls='--')
ax.axvline(9.0, color='gray', lw=0.7, ls='--')
ax.legend(fontsize=7)
ax_fmt(ax, ylabel='Accommodation (D)')
ax.set_title('Accommodation — direct blur drive + CA/C from vergence', fontsize=8)

ax = axes[3]
ax.plot(t3np, aca_drive3, color=C['aca'], lw=1.5, label='AC/A drive (acc→verg, deg)')
ax.axvline(2.0, color='gray', lw=0.7, ls='--')
ax.axvline(9.0, color='gray', lw=0.7, ls='--')
ax.legend(fontsize=7)
ax_fmt(ax, ylabel='AC/A vergence drive (deg)', xlabel='Time (s)')
ax.set_title('AC/A drive — rises with accommodation, contributes to vergence', fontsize=8)

fig.tight_layout()
plt.show()

# Steady-state breakdown
i_near = int(8.5 / DT)
disp_verg = vergence3[i_near] - aca_drive3[i_near]
print(f'\nSteady state near target ({Z_NEAR3}m):')
print(f'  Accommodation demand : {acc_demand_D(Z_NEAR3):.2f} D')
print(f'  x_acc total          : {acc_total3[i_near]:.2f} D')
print(f'  CA/C drive           : {cac_drive3[i_near]:.2f} D  '
      f'({100*cac_drive3[i_near]/max(acc_total3[i_near],0.01):.0f}% of total accommodation)')
print(f'  Vergence demand      : {verg_demand_deg(Z_NEAR3):.2f} deg')
print(f'  Actual vergence      : {vergence3[i_near]:.2f} deg')
print(f'  AC/A drive           : {aca_drive3[i_near]:.2f} deg  '
      f'({100*aca_drive3[i_near]/max(vergence3[i_near],0.01):.0f}% of total vergence)')

---
## Demo 4 — Closed-loop: accommodation + vergence co-tracking a depth step

Full interaction between both loops:

- **Disparity** drives vergence directly.
- **Vergence** state feeds CA/C → reduces effective blur demand → shapes accommodation.
- **Accommodation** state feeds AC/A → adds to vergence drive.

The **two loops are mutually stabilising** — Schor (1979) showed that isolated blur or
disparity stimulation drives both accommodation and vergence via these cross-links.

Here we verify that, for a natural depth step, both accommodation and vergence co-reach
the correct steady state, and that the cross-links speed up the approach.

In [ ]:
Z_STEPS = [(0.0, 2.0), (3.0, 0.4), (9.0, 1.0), (14.0, 0.2)]
T_END4  = 20.0

t4   = jnp.arange(0.0, T_END4, DT)
T4   = len(t4)
t4np = np.array(t4)

z4np = np.full(T4, Z_STEPS[0][1])
for t_on, z in Z_STEPS[1:]:
    z4np[t4np >= t_on] = z
z4 = jnp.array(z4np, dtype=jnp.float32)
pt4 = jnp.stack([jnp.zeros(T4), jnp.zeros(T4), z4], axis=1)

print('Simulating Demo 4 (closed-loop depth sequence)...')
st4 = simulate(
    THETA, t4,
    p_target_array      = pt4,
    scene_present_array = jnp.ones(T4),
    return_states       = True,
    key                 = jax.random.PRNGKey(0),
)
print('Done.')

x_acc4   = np.array(st4.brain[:, _IDX_ACC])
x_verg4  = np.array(st4.brain[:, _IDX_VERG])
ep_L4    = np.array(st4.plant[:, 0])
ep_R4    = np.array(st4.plant[:, 3])
vergence4 = ep_L4 - ep_R4
acc_total4 = x_acc4[:, 0] + x_acc4[:, 1]
acc_dem4   = acc_demand_D(z4np)
verg_dem4  = verg_demand_deg(z4np)

fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
fig.suptitle('Demo 4 — Closed-loop accommodation + vergence across depth steps', fontweight='bold')

# Shade depth steps
depth_colors = ['#f7f7f7', '#fde0dd', '#deebf7', '#fee8c8']
step_boundaries = [t for t, z in Z_STEPS] + [T_END4]
for i, ((t0, _), t1_) in enumerate(zip(Z_STEPS, step_boundaries[1:])):
    for ax in axes:
        ax.axvspan(t0, t1_, color=depth_colors[i % len(depth_colors)], alpha=0.4, zorder=0)

ax = axes[0]
ax.plot(t4np, z4np, color='k', lw=1.5, label='depth (m)')
for t_on, _ in Z_STEPS[1:]:
    ax.axvline(t_on, color='gray', lw=0.7, ls='--')
ax_fmt(ax, ylabel='Target depth (m)')
ax.set_title('Target depth sequence — 2→0.4→1→0.2 m', fontsize=8)

ax = axes[1]
ax.plot(t4np, verg_dem4,  color=C['dem'],   lw=1.2, ls='--', alpha=0.7, label='vergence demand (geometry)')
ax.plot(t4np, vergence4,  color=C['verg'],  lw=1.8,           label='vergence L−R (plant)')
ax.plot(t4np, x_verg4[:, 0], color=C['slow'], lw=0.9, ls=':', label='x_verg state')
for t_on, _ in Z_STEPS[1:]:
    ax.axvline(t_on, color='gray', lw=0.7, ls='--')
ax.legend(fontsize=7, loc='upper left')
ax_fmt(ax, ylabel='Vergence (deg)')
ax.set_title('Vergence — follows geometric demand at each depth', fontsize=8)

ax = axes[2]
ax.plot(t4np, acc_dem4,   color=C['dem'],   lw=1.2, ls='--', alpha=0.7, label='acc demand (1/z D)')
ax.plot(t4np, acc_total4, color=C['total'], lw=1.8,           label='x_acc total')
ax.plot(t4np, x_acc4[:, 0], color=C['fast'], lw=1.0, ls='-.',  label='x_fast (phasic)')
ax.plot(t4np, x_acc4[:, 1], color=C['slow'], lw=1.0, ls=':',   label='x_slow (tonic)')
for t_on, _ in Z_STEPS[1:]:
    ax.axvline(t_on, color='gray', lw=0.7, ls='--')
ax.legend(fontsize=7, loc='upper left')
ax_fmt(ax, ylabel='Accommodation (D)', xlabel='Time (s)')
ax.set_title('Accommodation — both components co-track vergence', fontsize=8)

fig.tight_layout()
plt.show()

print('\nSteady-state summary:')
for t_on, z in Z_STEPS:
    i = min(int((t_on + 2.5) / DT), T4 - 1)
    print(f'  z={z:.2f}m: verg demand={verg_demand_deg(z):.1f}°  '
          f'actual={vergence4[i]:.1f}°  '
          f'acc demand={acc_demand_D(z):.2f}D  total={acc_total4[i]:.2f}D')

---
## Summary

| Property | Model | Physiology |
|----------|-------|------------|
| Accommodation TC (fast) | τ ≈ 0.3 s | ~0.2–0.4 s [Schor 1979] |
| Accommodation TC (slow) | τ ≈ 30 s | ~20–60 s [Schor 1979] |
| AC/A ratio | 5 pd/D | 4–6 pd/D [Morgan 1944] |
| CA/C ratio | 0.4 D/pd | 0.3–0.5 D/pd [Fincham & Walton 1957] |
| Vergence TC (fusional) | ~160 ms | ~160 ms [Rashbass & Westheimer 1961] |
| Vergence integrator TC | τ = 6 s | 5–7 s [Semmlow et al. 1986] |

**AC/A clinical relevance**:
- High AC/A (>6) → esophoria at near (accommodative esotropia)
- Low AC/A (<3) → exophoria at near (convergence insufficiency)
- AC/A is the mechanism by which reading glasses (reducing accommodation demand) relieve accommodative esotropia

**CA/C clinical relevance**:
- Vergence training increases CA/C — improves focus maintenance during vergence
- Vision therapy for convergence insufficiency partially works by strengthening CA/C

### Model parameters to explore

```python
from oculomotor.sim.simulator import with_brain, PARAMS_DEFAULT

# High AC/A (accommodative esotropia)
theta_high_aca = with_brain(PARAMS_DEFAULT, AC_A=9.0)

# Low AC/A (convergence insufficiency)
theta_low_aca = with_brain(PARAMS_DEFAULT, AC_A=2.0)

# No CA/C cross-link
theta_no_cac = with_brain(PARAMS_DEFAULT, CA_C=0.0)

# Slow accommodation (presbyopia — loss of fast component amplitude)
theta_presbyopia = with_brain(PARAMS_DEFAULT, K_acc_fast=0.1, K_acc_slow=0.05)
```